# BCTC Annual PDF Manager

Notebook điều phối ingestion semi-structured (PDF-first) cho BCTC năm.

Luồng hiện tại chỉ 1 stage:
- Crawl danh sách công bố + download PDF + ghi metadata

Stage parse/OCR đã được loại bỏ khỏi flow này.

**Mặc định:** `hnx_verify_ssl=False` và `strict_bctc_annual_keyword_filter=False` — crawl `www.hnx.vn` và giữ mọi PDF để xem dữ liệu. Khi triển khai thật: bật `hnx_verify_ssl=True` (hoặc `HNX_SSL_VERIFY=1`) và có thể bật `strict_bctc_annual_keyword_filter=True` để chỉ BCTC năm.

**Số trang HNX:** code mặc định `hnx_max_list_pages=10` (DAG weekly). Notebook: `BCTC_RUN_PROFILE="backfill"` → **100** trang; `"dag_weekly"` → **10** (giống DAG, không cần set env).


In [1]:
import os
import sys
import warnings
import threading
from pathlib import Path

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

# SSL www.hnx.vn: mac dinh code dung hnx_verify_ssl=False. Muon bat verify TLS:
#   trong .env dat HNX_SSL_VERIFY=1  hoac  SemiStructuredIngestionConfig(hnx_verify_ssl=True)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook

def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        return
    _orig_hook(args)

threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("[OK] UTF-8 guard + sys.path ready")

[OK] UTF-8 guard + sys.path ready


In [ ]:
# backfill = lan dau / notebook: toi da 100 trang HNX (trang 1 = moi nhat)
# dag_weekly = giong mac dinh code (10 trang) — sau nay Airflow khong can set env
BCTC_RUN_PROFILE = "backfill"  # "backfill" | "dag_weekly"

_HNX_MAX_LIST_PAGES_BY_PROFILE = {
    "backfill": 100,
    "dag_weekly": 10,
}
if BCTC_RUN_PROFILE not in _HNX_MAX_LIST_PAGES_BY_PROFILE:
    raise ValueError(
        f"BCTC_RUN_PROFILE must be one of {list(_HNX_MAX_LIST_PAGES_BY_PROFILE)}, got {BCTC_RUN_PROFILE!r}"
    )
HNX_MAX_LIST_PAGES = _HNX_MAX_LIST_PAGES_BY_PROFILE[BCTC_RUN_PROFILE]
print(f"BCTC_RUN_PROFILE={BCTC_RUN_PROFILE} -> hnx_max_list_pages={HNX_MAX_LIST_PAGES}")

BCTC_RUN_PROFILE=backfill -> hnx_max_list_pages=500


In [3]:
import os

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.semi_structure_data import (
    SemiStructuredIngestionConfig,
    run_bctc_annual_pipeline,
)

configure_logging()
load_dotenv_from_project_root()

if os.environ.get("HNX_SSL_VERIFY", "").strip().lower() in ("0", "false", "no"):
    print("[INFO] .env: HNX_SSL_VERIFY — tat verify SSL cho www.hnx.vn")


cfg = SemiStructuredIngestionConfig(
    sources=["hnx"],
    hnx_max_list_pages=HNX_MAX_LIST_PAGES,
    rate_limit_rpm=20,
    min_pdf_bytes=20_000,
    # Mac dinh trong config: hnx_verify_ssl=False, strict_bctc_annual_keyword_filter=False
    # de crawl duoc tren Windows va xem du lieu. Production: bat True + HNX_SSL_VERIFY=1.
)

print(f"run_date: {cfg.run_date}")
print(f"data_lake_root: {cfg.data_lake_root}")
print(f"hnx_max_list_pages (config): {cfg.hnx_max_list_pages}")
print(f"hnx_max_list_pages (resolved): {cfg.resolved_hnx_max_list_pages()}")

2026-06-03 10:06:54 [INFO] Đã nạp biến môi trường từ D:\WorkSpace\Đồ Án 2\stock-pipeline\.env


run_date: 2026-06-03
data_lake_root: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw
hnx_max_list_pages (config): 500
hnx_max_list_pages (resolved): 500


In [4]:
# Download-only — tự bootstrap nếu chưa chạy cell import/`cfg`.
from pathlib import Path
import sys

_root = Path.cwd()
if not (_root / "ingestion").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.semi_structure_data import SemiStructuredIngestionConfig, run_bctc_annual_pipeline

configure_logging()
load_dotenv_from_project_root()

if "cfg" not in globals():
    _pages = globals().get("HNX_MAX_LIST_PAGES", 100)
    cfg = SemiStructuredIngestionConfig(sources=["hnx"], hnx_max_list_pages=_pages)

out_download = run_bctc_annual_pipeline(
    cfg,
    include_download=True,
 )
out_download

2026-06-03 10:06:54 [INFO] Đã nạp biến môi trường từ D:\WorkSpace\Đồ Án 2\stock-pipeline\.env
2026-06-03 10:06:54 [INFO] hnx_verify_ssl=False (mac dinh dev) — tat SSL verify cho www.hnx.vn; dat hnx_verify_ssl=True hoac HNX_SSL_VERIFY=1 khi can an toan
2026-06-03 10:06:54 [INFO] HNX list crawl: max_pages=500 start_page=1 rate_limit_rpm=20
2026-06-03 10:09:31 [INFO] HNX attachment accepted: SJ1 | 1.SJ1_2026.6.2_22381fe_EN_Reviewed_Consolidate_halfyear_financial_statements_Signed.pdf
2026-06-03 10:09:31 [INFO] HNX attachment accepted: SJ1 | 2.SJ1_2026.6.2_3a6e14f_VI_BCTC_ban_nien_HN_Signed.pdf
2026-06-03 10:09:54 [INFO] HNX attachment accepted: SAF | 1.SAF_2026.6.2_f0184c3_VI_thongbaoCBTT.pdf
2026-06-03 10:10:01 [INFO] HNX attachment accepted: PIC | 1.PIC_2026.6.2_5432ae9_VI_CBTTkhac.pdf
2026-06-03 10:10:01 [INFO] HNX attachment accepted: PIC | 2.PIC_2026.6.2_740d1ac_EN_otherextraordinarydisclosure.pdf
2026-06-03 10:11:37 [INFO] HNX attachment accepted: V21 | 1.V21_2026.6.1_1da6ef5_VI_CBT

{'run_date': '2026-06-03',
 'download': {'run_date': '2026-06-03',
  'documents_crawled': 1867,
  'download_ok': 1183,
  'download_fail': 0,
  'download_skipped_filter': 684,
  'metadata_parquet': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Semi_Structure_Data\\bctc_annual_pdf_meta\\source=hnx\\date=2026-06-03\\PART-000.parquet',
  'bootstrap_markers': ['D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Semi_Structure_Data\\bctc_annual_pdf\\source=hnx\\_full_bootstrap_done.json']}}

In [5]:
# Tổng quan dữ liệu đã tải (theo ticker)
from pathlib import Path
import pandas as pd

def _human_bytes(num: int) -> str:
    num = int(num or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num < 1024:
            return f"{num:.1f} {unit}" if unit != "B" else f"{num} {unit}"
        num /= 1024
    return f"{num:.1f} PB"

def _load_metadata_df() -> pd.DataFrame:
    if "out_download" in globals() and isinstance(out_download, dict):
        meta_path = out_download.get("metadata_parquet")
        if meta_path:
            p = Path(meta_path)
            if p.is_file():
                return pd.read_parquet(p)
    if "cfg" in globals():
        meta_root = cfg.bctc_pdf_meta_root / "source=hnx"
        if meta_root.is_dir():
            date_dirs = sorted([p for p in meta_root.iterdir() if p.is_dir()])
            if date_dirs:
                p = date_dirs[-1] / "PART-000.parquet"
                if p.is_file():
                    return pd.read_parquet(p)
    return pd.DataFrame()

df = _load_metadata_df()
if df.empty:
    print("Chưa tìm thấy metadata Parquet. Hãy chạy cell download trước.")
else:
    df["file_size"] = df["file_size"].fillna(0).astype(int)
    downloaded = df[df["status"] == "downloaded"].copy()
    if downloaded.empty:
        print("Không có file nào trạng thái 'downloaded' trong metadata.")
    else:
        summary = (
            downloaded.groupby("ticker", dropna=False)
            .agg(files=("pdf_path", "count"), total_bytes=("file_size", "sum"))
            .reset_index()
        )
        summary["total_size"] = summary["total_bytes"].apply(_human_bytes)
        summary = summary.sort_values(["total_bytes", "files"], ascending=False)
        display(summary)

        total_files = int(summary["files"].sum())
        total_bytes = int(summary["total_bytes"].sum())
        print(
            f"Tổng tickers: {summary.shape[0]} | Tổng files: {total_files} | "
            f"Tổng dung lượng: {_human_bytes(total_bytes)}"
        )

    status_counts = df["status"].value_counts(dropna=False)
    display(status_counts.rename_axis("status").reset_index(name="count"))

,ticker,files,total_bytes,total_size
45,DDG,25,177092703,168.9 MB
5,API,18,175247325,167.1 MB
48,DNP,12,168420844,160.6 MB
55,FID,11,163148751,155.6 MB
19,C69,10,161895935,154.4 MB
...,...,...,...,...
107,PMC,2,4806716,4.6 MB
109,PPE,2,2123064,2.0 MB
77,KSD,1,680247,664.3 KB
149,TKU,1,525299,513.0 KB


Tổng tickers: 189 | Tổng files: 1183 | Tổng dung lượng: 8.1 GB


,status,count
0,downloaded,1183
1,skipped_ingest_filter,684
